In [13]:
import pandas as pd
from pathlib import Path

# 📂 Carpeta base donde están tus archivos
BASE = Path(r"C:\Users\Lenovo\Desktop\LABO 3")

# 📥 Cargar predicciones
v1    = pd.read_csv(BASE / "submission_t780_autogluon_v1.csv").rename(columns={"tn": "tn_v1"})
v2    = pd.read_csv(BASE / "submission_t780_autogluon_v2.csv").rename(columns={"tn": "tn_v2"})
ens   = pd.read_csv(BASE / "submission_mega_ensemble.csv").rename(columns={"tn": "tn_ens"})
arima = pd.read_csv(BASE / "submission_t780_arima_win9.csv").rename(columns={"tn": "tn_arima"})

# 🔗 Merge por product_id
df = v1.merge(v2, on="product_id")
df = df.merge(ens, on="product_id")
df = df.merge(arima, on="product_id")

# 🧮 Ensemble con los pesos elegidos
df["tn"] = (
    0.4 * df["tn_v1"] +
    0.3 * df["tn_v2"] +
    0.2 * df["tn_ens"] +
    0.1 * df["tn_arima"]
)

# ✨ Postprocesar
q_hi = df["tn"].quantile(0.99)  # recorte del 1% más alto
df["tn"] = df["tn"].clip(lower=0, upper=q_hi)
df["tn"] = df["tn"].apply(lambda x: 0.5 if 0 < x < 0.5 else x)

# 💾 Guardar resultado final
OUT_PATH = BASE / "submission_hibrido_autogluon_priorizado.csv"
df[["product_id", "tn"]].to_csv(OUT_PATH, index=False, float_format="%.5f")

print(f"✅ Ensemble final guardado → {OUT_PATH}")

✅ Ensemble final guardado → C:\Users\Lenovo\Desktop\LABO 3\submission_hibrido_autogluon_priorizado.csv


In [9]:
import pandas as pd
from pathlib import Path

BASE = Path(r"C:\Users\Lenovo\Desktop\LABO 3")

# Cargar predicciones y renombrar
ens   = pd.read_csv(BASE / "submission_mega_ensemble.csv").rename(columns={"tn": "tn_ens"})
lgbm  = pd.read_csv(BASE / "submission_t780_lgbm_optuna.csv").rename(columns={"tn": "tn_lgbm"})
arima = pd.read_csv(BASE / "submission_t780_arima_win9.csv").rename(columns={"tn": "tn_arima"})
v1    = pd.read_csv(BASE / "submission_t780_autogluon_v1.csv").rename(columns={"tn": "tn_v1"})

# Merge
df = ens.merge(lgbm, on="product_id")
df = df.merge(arima, on="product_id")
df = df.merge(v1, on="product_id")

# Ensemble con nuevos pesos más robustos
df["tn"] = (
    0.4 * df["tn_ens"] +
    0.3 * df["tn_lgbm"] +
    0.2 * df["tn_arima"] +
    0.1 * df["tn_v1"]
)

# Postprocesar
q_hi = df["tn"].quantile(0.99)
df["tn"] = df["tn"].clip(lower=0, upper=q_hi)
df["tn"] = df["tn"].apply(lambda x: 0.5 if 0 < x < 0.5 else x)

# Guardar
OUT_PATH = BASE / "submission_hibrido_ensemble_priorizado.csv"
df[["product_id", "tn"]].to_csv(OUT_PATH, index=False, float_format="%.5f")

print(f"✅ Ensemble final guardado → {OUT_PATH}")

✅ Ensemble final guardado → C:\Users\Lenovo\Desktop\LABO 3\submission_hibrido_ensemble_priorizado.csv


In [11]:
import pandas as pd
from pathlib import Path

# 📂 Carpeta base
BASE = Path(r"C:\Users\Lenovo\Desktop\LABO 3")

# 📥 Cargar predicciones y renombrar columnas tn
v1    = pd.read_csv(BASE / "submission_t780_autogluon_v1.csv").rename(columns={"tn": "tn_v1"})
mega  = pd.read_csv(BASE / "submission_mega_ensemble.csv").rename(columns={"tn": "tn_mega"})
lgbm  = pd.read_csv(BASE / "submission_t780_lgbm_optuna.csv").rename(columns={"tn": "tn_lgbm"})
arima = pd.read_csv(BASE / "submission_t780_arima_win9.csv").rename(columns={"tn": "tn_arima"})

# 🔗 Combinar por product_id
df = v1.merge(mega, on="product_id")
df = df.merge(lgbm, on="product_id")
df = df.merge(arima, on="product_id")

# 🧮 Calcular ensemble con las ponderaciones finales
df["tn"] = (
    0.30 * df["tn_v1"] +
    0.30 * df["tn_mega"] +
    0.25 * df["tn_lgbm"] +
    0.15 * df["tn_arima"]
)

# ✨ Postprocesamiento: recorte y suavizado
q_hi = df["tn"].quantile(0.99)
df["tn"] = df["tn"].clip(lower=0, upper=q_hi)
df["tn"] = df["tn"].apply(lambda x: 0.5 if 0 < x < 0.5 else x)

# 💾 Guardar archivo final
OUT_PATH = BASE / "submission_hibrido_dif_pesos.csv"
df[["product_id", "tn"]].to_csv(OUT_PATH, index=False, float_format="%.5f")

print(f"✅ Archivo final guardado en: {OUT_PATH}")

✅ Archivo final guardado en: C:\Users\Lenovo\Desktop\LABO 3\submission_hibrido_dif_pesos.csv
